# Auditing components.yaml

This notebook runs the AuditSystem against the main components.yaml file to evaluate solution quality.

In [ ]:
from pathlib import Path

import ibis as ib
from ibis import _

from iacs.architect import Architect

ib.options.interactive = True  # auto-display results (like pandas)

## Load components.yaml

In [ ]:
COMPONENTS_DIR = "../builtins"

a = Architect.from_manifest(COMPONENTS_DIR)

## Evaluate highest-priority tasks to work on

In [ ]:
entity_ids = a.get("entity_id")
entity_ids

In [ ]:
parents = a.get("parent")
parents

In [ ]:
reqs = a.get("requirement")
reqs

In [ ]:
(
    reqs
    .join(parents, reqs.entity_id == parents.parent_id, how="anti")
    .join(entity_ids.select(_.value, _.entity_key, _.path), _.entity_id == entity_ids.value )
    .order_by(ib.desc(_.priority))
)

## Requirements tree visualization

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

import networkx as nx

# Configure
ANCESTOR_KEY = "be_a_powerful_tool_for_solutions_architecture"
OUTPUT_PATH = Path("../notebooks/tree_data.json")

# Pull data to pandas
entity_ids_pd = entity_ids.to_pandas()
parents_pd = parents.to_pandas()
reqs_pd = reqs.to_pandas()

id_to_key = entity_ids_pd.set_index("value")["entity_key"].to_dict()
req_ids = set(reqs_pd["entity_id"].unique())

ancestor_rows = entity_ids_pd[entity_ids_pd["entity_key"] == ANCESTOR_KEY]
if ancestor_rows.empty:
    raise ValueError(f"No entity found with entity_key '{ANCESTOR_KEY}'")
ancestor_id = ancestor_rows.iloc[0]["value"]

# Build full graph and find req descendants
G_full = nx.DiGraph()
for _, row in parents_pd.iterrows():
    G_full.add_edge(row["parent_id"], row["entity_id"])

descendants = nx.descendants(G_full, ancestor_id) | {ancestor_id}
req_nodes = (descendants & req_ids) | {ancestor_id}

# Build children lookup restricted to req_nodes
children_map = defaultdict(list)
for _, row in parents_pd.iterrows():
    if row["parent_id"] in req_nodes and row["entity_id"] in req_nodes:
        children_map[row["parent_id"]].append(row["entity_id"])

def build_tree(node_id):
    node = {"name": id_to_key.get(node_id, node_id[:8])}
    children = children_map.get(node_id, [])
    if children:
        node["children"] = [build_tree(c) for c in children]
    return node

tree_data = build_tree(ancestor_id)
OUTPUT_PATH.write_text(json.dumps(tree_data, indent=2))
print(f"Wrote {OUTPUT_PATH}")